# Module 11 · 2：文字下游 — 分類微調 / LLM LoRA / RAG

> 文字資料（M09 文字小節）整理好後，三條最常見的下游路線：
> **(A) 微調分類器**、**(B) LLM 指令微調 (LoRA/SFT)**、**(C) 檢索增強生成 (RAG)**。

## 學習目標
- 用 `Trainer` 微調 **DistilBERT** 文本分類器（CPU 小 demo 設定）。
- 認識 **LLM 指令微調** 的資料格式與 `peft` (LoRA) + `trl` (SFT) 最小設定。
- 用句向量做一個最小 **RAG 檢索** demo。

In [ ]:
try:
    import torch
    HAS = True
except Exception:
    HAS = False
    print("需 `uv sync --extra multimodal --extra train`。說明仍可閱讀。")

## A. 微調 DistilBERT 文本分類器

資料格式：`input_ids (B,L)` + 標籤 `(B,)`。用 `Trainer` 包好訓練迴圈。
下方為 **CPU 友善的最小設定**（少量樣本、1 epoch）；首次執行會下載 DistilBERT。

In [ ]:
if HAS:
    try:
        from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                                   TrainingArguments, Trainer)
        from datasets import Dataset

        texts = ["great movie", "loved it", "fantastic", "terrible", "boring", "awful"]
        labels = [1, 1, 1, 0, 0, 0]
        ds = Dataset.from_dict({"text": texts, "label": labels})

        tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")
        ds = ds.map(lambda b: tok(b["text"], truncation=True, padding="max_length", max_length=16),
                    batched=True)

        model = AutoModelForSequenceClassification.from_pretrained(
            "distilbert-base-uncased", num_labels=2)

        args = TrainingArguments(
            output_dir="/tmp/distilbert_demo",
            num_train_epochs=1,            # demo：1 epoch
            per_device_train_batch_size=2,
            learning_rate=5e-5,
            logging_steps=1,
            report_to=[],                  # 不上傳任何追蹤平台
            no_cuda=True,                  # 強制 CPU（demo）
        )
        trainer = Trainer(model=model, args=args, train_dataset=ds)
        trainer.train()
        print("✓ DistilBERT 分類器最小微調完成（demo）。")
        print("  實務：用更大資料 + 適當 epoch + 驗證集評估。")
    except Exception as e:
        print("（未能下載 DistilBERT 或環境不足，略過實際訓練）", e)

## B. LLM 指令微調（LoRA + SFT）

要把一個基礎 LLM 變成「會聽指令的助理」，用 **SFT（監督式微調）**；
為了省算力，用 **LoRA** 只訓練極少參數。資料是 `04_llm_data_formats` 的 chat/instruction JSONL。

下面展示**最小設定**（不在 CPU 跑完整訓練，重點在資料格式與設定）。

In [ ]:
if HAS:
    try:
        from peft import LoraConfig
        from trl import SFTConfig

        # LoRA 設定：只在注意力投影層插入低秩矩陣
        lora_cfg = LoraConfig(
            r=16, lora_alpha=32, lora_dropout=0.05,
            target_modules=["q_proj", "v_proj"],
            task_type="CAUSAL_LM",
        )
        # SFT 設定（trl）
        sft_cfg = SFTConfig(
            output_dir="/tmp/llm_lora_demo",
            num_train_epochs=1,
            per_device_train_batch_size=1,
            learning_rate=2e-4,
            max_length=512,
        )
        # 訓練資料：chat 格式（與 M09 文字 04 一致）
        train_data = [{"messages": [
            {"role": "user", "content": "什麼是 LoRA？"},
            {"role": "assistant", "content": "LoRA 是一種只訓練少量新增低秩矩陣的高效微調法。"},
        ]}]
        print("LoRA 設定:", lora_cfg.r, lora_cfg.target_modules)
        print("SFT 設定 lr:", sft_cfg.learning_rate)
        print("訓練資料格式: chat messages（見 M09 文字 04）")
        print("\n實際訓練：trl 的 SFTTrainer(model, args=sft_cfg, train_dataset=..., peft_config=lora_cfg).train()")
        print("可訓練參數通常 < 1%，單張消費級 GPU（甚至 QLoRA 4-bit）即可微調 7B 模型。")
    except Exception as e:
        print("（未安裝 peft/trl，略過）", e)

## C. RAG：檢索增強生成（最小檢索 demo）

RAG = **檢索**（用句向量找到相關文件）+ **生成**（把找到的內容塞進 LLM prompt）。
這裡示範「檢索」這一半（生成那半交給任一 LLM）。

In [ ]:
if HAS:
    try:
        from sentence_transformers import SentenceTransformer, util
        st = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
        knowledge = [
            "LoRA 透過低秩矩陣高效微調大模型。",
            "Whisper 是 OpenAI 的語音辨識模型。",
            "RAG 把檢索到的文件加入提示來增強生成。",
            "ViT 把影像切成 patch 當作 token。",
        ]
        kb_emb = st.encode(knowledge, convert_to_tensor=True)
        query = "怎麼用很少的算力微調大型語言模型？"
        q = st.encode(query, convert_to_tensor=True)
        hit = util.cos_sim(q, kb_emb)[0].argmax().item()
        print(f"問題: {query}")
        print(f"檢索到最相關知識: {knowledge[hit]}")
        print("→ 接著把這段文字放進 LLM 的 prompt（context），即可做出有依據的回答。")
    except Exception as e:
        print("（未安裝 sentence-transformers，略過）", e)

## 小結

| 路線 | 資料格式 | 模型 | 技術 |
|:--|:--|:--|:--|
| A 分類微調 | `input_ids (B,L)` + 標籤 | DistilBERT | `Trainer` |
| B 指令微調 | chat/instruction JSONL | LLM | `peft` LoRA + `trl` SFT |
| C RAG | 句向量 + 知識庫 | 檢索器 + LLM | 語意檢索 |

文字是大模型應用最成熟的模態，三條路線涵蓋了多數實務需求。